
# Data Modeling: From Concepts to Star & Snowflake Schemas

**Audience:** Data engineers, analytics engineers, BI developers, students.  
**Goals:**
- Understand what data modeling is and why it matters.
- Learn the differences between conceptual, logical, and physical models.
- See how to convert a transactional dataset into a dimensional model.
- Build a **Star Schema** and a **Snowflake Schema** for a retail example.
- Inspect sample SQL DDL for creating these structures.



## 1. What is Data Modeling?

**Data modeling** is the process of designing a blueprint for how data is **structured, related, and governed** in an information system. It aligns **business rules** with **technical implementation**, ensuring the data is **accurate, consistent, and efficient** to query.

### 1.1 Why it Matters
- **Integrity & Quality:** Clear keys, constraints, and relationships reduce ambiguity and duplication.
- **Performance:** Proper structures and indexes speed up queries and ETL/ELT processes.
- **Governance & Evolution:** Shared vocabulary and documentation make change safer.
- **Analytics Readiness:** Makes BI and ML workloads easier (aggregation, slicing, time-series).

### 1.2 Model Layers
- **Conceptual Model** – high‑level picture of entities and relationships (business view).
- **Logical Model** – attributes, keys, relationships (technology-agnostic).
- **Physical Model** – tables, columns, data types, partitions, indexes (platform-specific).

### 1.3 Common Approaches
- **Entity‑Relationship (ER) Modeling** for OLTP systems (3NF focus).
- **Dimensional Modeling** (Kimball) for analytics/warehousing – facts & dimensions.
- **Data Vault** for large, evolving warehouses with lineage & historization needs.

### 1.4 Normalization vs. Denormalization
- **Normalization (1NF, 2NF, 3NF, …)** minimizes redundancy and anomalies; ideal for **OLTP**.
- **Denormalization** purposely duplicates certain attributes for **fast reads**; typical for **analytics**.



## 2. Dimensional Modeling (Analytics-Focused)

### 2.1 Core Ideas
- **Facts**: Numeric, additive measures about a business event (e.g., `sales_amount`, `quantity`).
- **Dimensions**: Descriptive context to slice facts (e.g., `product`, `customer`, `date`, `store`).
- **Grain**: The *atomic* level of detail of a fact row (e.g., one order line). Choosing grain first is crucial.
- **Keys**: Surrogate keys are commonly used for dimensions; fact tables store foreign keys to dimensions.

### 2.2 Star vs. Snowflake
- **Star Schema**: One central fact table connected to **denormalized** dimension tables.
- **Snowflake Schema**: Dimensions are **normalized** into sub-dimensions (e.g., Product → Category).

Both are valid; **Star** is simpler and often faster for BI. **Snowflake** can reduce storage and improve maintenance of hierarchical attributes.



## 3. Example Transactions (Retail)

We will model a simplified online retail process with an **order line** grain.

**Raw transactional table** (typical OLTP export):

| order_id | order_line_id | order_ts           | customer_name | customer_city | customer_region | product_sku | product_name | product_category | store_name      | store_region | quantity | unit_price | discount_amount | payment_type |
|----------|----------------|--------------------|---------------|---------------|-----------------|-------------|--------------|------------------|-----------------|--------------|----------|------------|-----------------|--------------|
| 1001     | 1              | 2025-10-15 10:05   | John Doe      | Dallas        | South           | LTP-15      | Laptop 15"   | Electronics      | Dallas Downtown | South        | 1        | 900        | 0               | Credit Card  |
| 1002     | 1              | 2025-10-16 14:22   | Mary Ann      | Austin        | South           | SHO-01      | Running Shoe | Apparel          | Austin Central  | South        | 2        | 60         | 0               | PayPal       |

We will synthesize a small dataset to demonstrate the transformations.


In [5]:

# Create a small synthetic transactional dataset
import pandas as pd

raw = pd.DataFrame([
    {"order_id":1001,"order_line_id":1,"order_ts":"2025-10-15 10:05","customer_name":"John Doe","customer_city":"Dallas","customer_region":"South","product_sku":"LTP-15","product_name":"Laptop 15\"","product_category":"Electronics","store_name":"Dallas Downtown","store_region":"South","quantity":1,"unit_price":900.0,"discount_amount":0.0,"payment_type":"Credit Card"},
    {"order_id":1002,"order_line_id":1,"order_ts":"2025-10-16 14:22","customer_name":"Mary Ann","customer_city":"Austin","customer_region":"South","product_sku":"SHO-01","product_name":"Running Shoe","product_category":"Apparel","store_name":"Austin Central","store_region":"South","quantity":2,"unit_price":60.0,"discount_amount":0.0,"payment_type":"PayPal"},
    {"order_id":1002,"order_line_id":2,"order_ts":"2025-10-16 14:25","customer_name":"Mary Ann","customer_city":"Austin","customer_region":"South","product_sku":"SOX-01","product_name":"Socks","product_category":"Apparel","store_name":"Austin Central","store_region":"South","quantity":3,"unit_price":5.0,"discount_amount":0.0,"payment_type":"PayPal"},
    {"order_id":1003,"order_line_id":1,"order_ts":"2025-10-17 09:10","customer_name":"John Doe","customer_city":"Dallas","customer_region":"South","product_sku":"MOU-01","product_name":"Wireless Mouse","product_category":"Electronics","store_name":"Online","store_region":"US","quantity":1,"unit_price":25.0,"discount_amount":5.0,"payment_type":"Credit Card"}
])

raw["order_ts"] = pd.to_datetime(raw["order_ts"])
raw["extended_price"] = raw["quantity"] * raw["unit_price"] - raw["discount_amount"]
raw


,order_id,order_line_id,order_ts,customer_name,customer_city,customer_region,product_sku,product_name,product_category,store_name,store_region,quantity,unit_price,discount_amount,payment_type,extended_price
0,1001,1,2025-10-15 10:05:00,John Doe,Dallas,South,LTP-15,"Laptop 15""",Electronics,Dallas Downtown,South,1,900.0,0.0,Credit Card,900.0
1,1002,1,2025-10-16 14:22:00,Mary Ann,Austin,South,SHO-01,Running Shoe,Apparel,Austin Central,South,2,60.0,0.0,PayPal,120.0
2,1002,2,2025-10-16 14:25:00,Mary Ann,Austin,South,SOX-01,Socks,Apparel,Austin Central,South,3,5.0,0.0,PayPal,15.0
3,1003,1,2025-10-17 09:10:00,John Doe,Dallas,South,MOU-01,Wireless Mouse,Electronics,Online,US,1,25.0,5.0,Credit Card,20.0


In [6]:

# from caas_jupyter_tools import display_dataframe_to_user
# display_dataframe_to_user("Raw Retail Transactions", raw)
display(raw.head())   # or print(raw.head())


,order_id,order_line_id,order_ts,customer_name,customer_city,customer_region,product_sku,product_name,product_category,store_name,store_region,quantity,unit_price,discount_amount,payment_type,extended_price
0,1001,1,2025-10-15 10:05:00,John Doe,Dallas,South,LTP-15,"Laptop 15""",Electronics,Dallas Downtown,South,1,900.0,0.0,Credit Card,900.0
1,1002,1,2025-10-16 14:22:00,Mary Ann,Austin,South,SHO-01,Running Shoe,Apparel,Austin Central,South,2,60.0,0.0,PayPal,120.0
2,1002,2,2025-10-16 14:25:00,Mary Ann,Austin,South,SOX-01,Socks,Apparel,Austin Central,South,3,5.0,0.0,PayPal,15.0
3,1003,1,2025-10-17 09:10:00,John Doe,Dallas,South,MOU-01,Wireless Mouse,Electronics,Online,US,1,25.0,5.0,Credit Card,20.0



## 4. Build a Star Schema (Order Line Grain)

**Grain**: One row per **order line**.

We will create the following:
- `dim_date`: date attributes derived from `order_ts`
- `dim_customer`: customer attributes
- `dim_product`: product attributes
- `dim_store`: store attributes
- `fact_sales`: numeric measures and foreign keys to the dimensions


In [7]:

# Build STAR SCHEMA tables (simple surrogate keys via integer enumerations)

# dim_date
date_df = raw[["order_ts"]].copy()
date_df["date"] = date_df["order_ts"].dt.date
date_df["year"] = date_df["order_ts"].dt.year
date_df["month"] = date_df["order_ts"].dt.month
date_df["day"] = date_df["order_ts"].dt.day
date_df["quarter"] = ((date_df["month"]-1)//3 + 1)
date_df = date_df.drop_duplicates(subset=["date"]).reset_index(drop=True)
date_df = date_df[["date","year","quarter","month","day"]].copy()
date_df.insert(0, "date_key", range(1, len(date_df)+1))

# dim_customer
dim_customer = raw[["customer_name","customer_city","customer_region"]].drop_duplicates().reset_index(drop=True)
dim_customer.insert(0, "customer_key", range(1, len(dim_customer)+1))

# dim_product
dim_product = raw[["product_sku","product_name","product_category"]].drop_duplicates().reset_index(drop=True)
dim_product.insert(0, "product_key", range(1, len(dim_product)+1))

# dim_store
dim_store = raw[["store_name","store_region"]].drop_duplicates().reset_index(drop=True)
dim_store.insert(0, "store_key", range(1, len(dim_store)+1))

# Helper maps
date_map = dict(zip(date_df["date"], date_df["date_key"]))
cust_map = dict(zip(dim_customer["customer_name"] + "|" + dim_customer["customer_city"] + "|" + dim_customer["customer_region"],
                    dim_customer["customer_key"]))
prod_map = dict(zip(dim_product["product_sku"], dim_product["product_key"]))
store_map = dict(zip(dim_store["store_name"] + "|" + dim_store["store_region"], dim_store["store_key"]))

# fact_sales
fact_sales = raw.copy()
fact_sales["date"] = fact_sales["order_ts"].dt.date

fact_sales["customer_key"] = fact_sales["customer_name"] + "|" + fact_sales["customer_city"] + "|" + fact_sales["customer_region"]
fact_sales["customer_key"] = fact_sales["customer_key"].map(cust_map)

fact_sales["product_key"] = fact_sales["product_sku"].map(prod_map)
fact_sales["store_key"] = (fact_sales["store_name"] + "|" + fact_sales["store_region"]).map(store_map)
fact_sales["date_key"] = fact_sales["date"].map(date_map)

fact_sales = fact_sales[[
    "order_id","order_line_id","date_key","customer_key","product_key","store_key",
    "quantity","unit_price","discount_amount","extended_price"
]].rename(columns={
    "extended_price":"sales_amount"
}).sort_values(["order_id","order_line_id"]).reset_index(drop=True)

date_df, dim_customer, dim_product, dim_store, fact_sales


(   date_key        date  year  quarter  month  day
 0         1  2025-10-15  2025        4     10   15
 1         2  2025-10-16  2025        4     10   16
 2         3  2025-10-17  2025        4     10   17,
    customer_key customer_name customer_city customer_region
 0             1      John Doe        Dallas           South
 1             2      Mary Ann        Austin           South,
    product_key product_sku    product_name product_category
 0            1      LTP-15      Laptop 15"      Electronics
 1            2      SHO-01    Running Shoe          Apparel
 2            3      SOX-01           Socks          Apparel
 3            4      MOU-01  Wireless Mouse      Electronics,
    store_key       store_name store_region
 0          1  Dallas Downtown        South
 1          2   Austin Central        South
 2          3           Online           US,
    order_id  order_line_id  date_key  customer_key  product_key  store_key  \
 0      1001              1         1        

In [8]:
# Simply use the built-in display() or print() instead
display(date_df)
display(dim_customer)
display(dim_product)
display(dim_store)
display(fact_sales)


ModuleNotFoundError: No module named 'caas_jupyter_tools'


### 4.1 Visual: Star Schema

A simple diagram showing the central fact table linked to dimensions.


In [ ]:

# Draw a simple star schema figure using matplotlib (no seaborn, single plot, no explicit colors)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

# Boxes: fact + 4 dims
# (x, y, width, height)
boxes = {
    "fact_sales": (3.5, 3.2, 3.0, 1.0),
    "dim_date": (0.5, 5.0, 2.8, 0.9),
    "dim_customer": (0.5, 1.8, 3.0, 0.9),
    "dim_product": (7.0, 5.0, 3.0, 0.9),
    "dim_store": (7.0, 1.8, 3.0, 0.9)
}

def draw_box(ax, name, rect, title):
    x,y,w,h = rect
    ax.add_patch(plt.Rectangle((x,y), w, h, fill=False))
    ax.text(x+w/2, y+h/2, title, ha='center', va='center', fontsize=10)

fig = plt.gcf()
ax = plt.gca()
for k, rect in boxes.items():
    title = k.replace("_", " ").title()
    draw_box(ax, k, rect, title)

# Connectors
def connect(ax, a, b):
    ax.plot([a[0]+a[2]/2, b[0]+b[2]/2], [a[1], b[1]+b[3]], linewidth=1)

connect(ax, boxes["fact_sales"], boxes["dim_date"])
connect(ax, boxes["fact_sales"], boxes["dim_customer"])
connect(ax, boxes["fact_sales"], boxes["dim_product"])
connect(ax, boxes["fact_sales"], boxes["dim_store"])

ax.set_xlim(0, 11)
ax.set_ylim(0, 7.5)
ax.axis('off')
plt.title("Star Schema: fact_sales linked to dim_date, dim_customer, dim_product, dim_store")
plt.show()



## 5. Convert to a Snowflake Schema

We'll **normalize** some dimensions to show a snowflake pattern:
- Split `dim_product` into `dim_product` + `dim_product_category`.
- Split `dim_date` into `dim_date` + `dim_month` (toy example for hierarchy).

This reduces duplication of repeated attributes (e.g., many products share a category).


In [ ]:

# Create sub-dimensions for a simple snowflake pattern

# dim_product_category
dim_product_category = dim_product[["product_category"]].drop_duplicates().reset_index(drop=True)
dim_product_category.insert(0, "product_category_key", range(1, len(dim_product_category)+1))

# Relate product -> category via surrogate FK
cat_map = dict(zip(dim_product_category["product_category"], dim_product_category["product_category_key"]))
dim_product_snow = dim_product.copy()
dim_product_snow["product_category_key"] = dim_product_snow["product_category"].map(cat_map)
dim_product_snow = dim_product_snow.drop(columns=["product_category"])

# dim_month for date hierarchy example
dim_month = date_df[["year","month","quarter"]].drop_duplicates().reset_index(drop=True)
dim_month.insert(0, "month_key", range(1, len(dim_month)+1))

# relate date -> month
month_map = dict(zip(zip(dim_month["year"], dim_month["month"]), dim_month["month_key"]))
dim_date_snow = date_df.copy()
dim_date_snow["month_key"] = list(map(lambda r: month_map[(r["year"], r["month"])], dim_date_snow.to_dict("records")))
dim_date_snow = dim_date_snow.drop(columns=["quarter","month","day"])

dim_product_category, dim_product_snow, dim_month, dim_date_snow


In [ ]:

from caas_jupyter_tools import display_dataframe_to_user
display_dataframe_to_user("dim_product_category", dim_product_category)
display_dataframe_to_user("dim_product (snowflaked)", dim_product_snow)
display_dataframe_to_user("dim_month", dim_month)
display_dataframe_to_user("dim_date (snowflaked)", dim_date_snow)



### 5.1 Visual: Snowflake (Product & Date Hierarchies)


In [ ]:

# Draw a simple snowflake diagram focusing on product and date hierarchies
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 6))
ax = plt.gca()

# Core fact and dims
boxes = {
    "fact_sales": (3.8, 3.0, 3.0, 1.0),
    "dim_customer": (0.6, 1.2, 3.0, 0.9),
    "dim_store": (7.2, 1.2, 3.0, 0.9),
    "dim_product_snow": (7.2, 4.6, 3.0, 0.9),
    "dim_product_category": (7.2, 6.0, 3.0, 0.9),
    "dim_date_snow": (0.6, 4.6, 3.0, 0.9),
    "dim_month": (0.6, 6.0, 3.0, 0.9)
}

def draw_box(ax, rect, title):
    x,y,w,h = rect
    ax.add_patch(plt.Rectangle((x,y), w, h, fill=False))
    ax.text(x+w/2, y+h/2, title, ha='center', va='center', fontsize=10)

for title, rect in boxes.items():
    draw_box(ax, rect, title.replace("_", " ").title())

# connections
def connect_bottom_to_top(ax, bottom, top):
    ax.plot([bottom[0]+bottom[2]/2, top[0]+top[2]/2], [bottom[1]+bottom[3], top[1]], linewidth=1)

def connect(ax, a, b):
    ax.plot([a[0]+a[2]/2, b[0]+b[2]/2], [a[1], b[1]+b[3]], linewidth=1)

connect(ax, boxes["fact_sales"], boxes["dim_customer"])
connect(ax, boxes["fact_sales"], boxes["dim_store"])
connect(ax, boxes["fact_sales"], boxes["dim_product_snow"])
connect(ax, boxes["fact_sales"], boxes["dim_date_snow"])

connect_bottom_to_top(ax, boxes["dim_product_snow"], boxes["dim_product_category"])
connect_bottom_to_top(ax, boxes["dim_date_snow"], boxes["dim_month"])

ax.set_xlim(0, 11.5)
ax.set_ylim(0.5, 7.5)
ax.axis('off')
plt.title("Snowflake: Normalized Product & Date Dimensions")
plt.show()



## 6. Example SQL DDL

Below are simplified DDL snippets for a **Star Schema** and its **Snowflake** variant.  
Adjust data types and engine-specific syntax for Snowflake, BigQuery, Redshift, Databricks, etc.

### 6.1 Star Schema (DDL)


In [ ]:
star_schema_sql = r'''
-- Dimension tables
CREATE TABLE dim_date (
  date_key        INT PRIMARY KEY,
  date            DATE NOT NULL,
  year            INT NOT NULL,
  quarter         INT NOT NULL,
  month           INT NOT NULL,
  day             INT NOT NULL
);

CREATE TABLE dim_customer (
  customer_key    INT PRIMARY KEY,
  customer_name   VARCHAR(200),
  customer_city   VARCHAR(200),
  customer_region VARCHAR(100)
);

CREATE TABLE dim_product (
  product_key     INT PRIMARY KEY,
  product_sku     VARCHAR(50) UNIQUE,
  product_name    VARCHAR(200),
  product_category VARCHAR(100)
);

CREATE TABLE dim_store (
  store_key       INT PRIMARY KEY,
  store_name      VARCHAR(200),
  store_region    VARCHAR(100)
);

-- Fact table
CREATE TABLE fact_sales (
  order_id        BIGINT,
  order_line_id   INT,
  date_key        INT REFERENCES dim_date(date_key),
  customer_key    INT REFERENCES dim_customer(customer_key),
  product_key     INT REFERENCES dim_product(product_key),
  store_key       INT REFERENCES dim_store(store_key),
  quantity        INT,
  unit_price      NUMERIC(12,2),
  discount_amount NUMERIC(12,2),
  sales_amount    NUMERIC(12,2),
  PRIMARY KEY (order_id, order_line_id)
);
'''
print(star_schema_sql)


### 6.2 Snowflake Schema (DDL)
We split `dim_product` into `dim_product` + `dim_product_category`, and `dim_date` into `dim_date` + `dim_month`.


In [ ]:
snowflake_schema_sql = r'''
-- Product hierarchy
CREATE TABLE dim_product_category (
  product_category_key INT PRIMARY KEY,
  product_category     VARCHAR(100) UNIQUE
);

CREATE TABLE dim_product (
  product_key          INT PRIMARY KEY,
  product_sku          VARCHAR(50) UNIQUE,
  product_name         VARCHAR(200),
  product_category_key INT REFERENCES dim_product_category(product_category_key)
);

-- Date hierarchy
CREATE TABLE dim_month (
  month_key            INT PRIMARY KEY,
  year                 INT NOT NULL,
  month                INT NOT NULL,
  quarter              INT NOT NULL
);

CREATE TABLE dim_date (
  date_key             INT PRIMARY KEY,
  date                 DATE NOT NULL,
  month_key            INT REFERENCES dim_month(month_key)
);

-- Other dimensions remain the same as star

-- Fact table remains the same foreign-key-wise (to dim_date, dim_product, etc.)
CREATE TABLE fact_sales (
  order_id        BIGINT,
  order_line_id   INT,
  date_key        INT REFERENCES dim_date(date_key),
  customer_key    INT,
  product_key     INT REFERENCES dim_product(product_key),
  store_key       INT,
  quantity        INT,
  unit_price      NUMERIC(12,2),
  discount_amount NUMERIC(12,2),
  sales_amount    NUMERIC(12,2),
  PRIMARY KEY (order_id, order_line_id)
);
'''
print(snowflake_schema_sql)


## 7. Query Examples (Analytical)

- **Monthly Revenue by Product Category:**
```sql
SELECT d.year, d.month, p.product_category, SUM(f.sales_amount) AS revenue
FROM fact_sales f
JOIN dim_date d     ON f.date_key = d.date_key
JOIN dim_product p  ON f.product_key = p.product_key
GROUP BY d.year, d.month, p.product_category
ORDER BY d.year, d.month, p.product_category;
```

- **Top 5 Customers by Revenue (Last Quarter):**
```sql
SELECT c.customer_name, SUM(f.sales_amount) AS revenue
FROM fact_sales f
JOIN dim_customer c ON f.customer_key = c.customer_key
JOIN dim_date d     ON f.date_key = d.date_key
WHERE d.year = 2025 AND d.quarter = 4
GROUP BY c.customer_name
ORDER BY revenue DESC
LIMIT 5;
```

- **Average Order Value per Store:**
```sql
SELECT s.store_name, AVG(f.sales_amount) AS avg_order_value
FROM fact_sales f
JOIN dim_store s ON f.store_key = s.store_key
GROUP BY s.store_name
ORDER BY avg_order_value DESC;
```



## 8. Best Practices & Tips

- **Fix the grain early** for each fact table (e.g., *order line*, not *order*).
- Prefer **additive or semi-additive** measures for robust rollups; document non-additive metrics.
- Use **surrogate keys** for slowly changing dimensions (SCD), especially Type 2 for history.
- Keep **dimensions conformed** across facts to enable cross-process analytics.
- For cloud DWs, leverage **clustering/partitioning** and **columnar storage**.
- Validate with **sample queries** and BI tools to ensure model usability.
- Document with **data dictionaries** and **ERD/lineage** diagrams.



## 9. (Optional) Export Sample Tables as CSV

Run the next cell to export the **dim** and **fact** tables as CSVs for downstream experiments.


In [ ]:

import os

out_dir = "/mnt/data/data_modeling_outputs"
os.makedirs(out_dir, exist_ok=True)

date_df.to_csv(f"{out_dir}/dim_date.csv", index=False)
dim_customer.to_csv(f"{out_dir}/dim_customer.csv", index=False)
dim_product.to_csv(f"{out_dir}/dim_product.csv", index=False)
dim_store.to_csv(f"{out_dir}/dim_store.csv", index=False)
fact_sales.to_csv(f"{out_dir}/fact_sales.csv", index=False)

dim_product_category.to_csv(f"{out_dir}/dim_product_category.csv", index=False)
dim_product_snow.to_csv(f"{out_dir}/dim_product_snow.csv", index=False)
dim_month.to_csv(f"{out_dir}/dim_month.csv", index=False)
dim_date_snow.to_csv(f"{out_dir}/dim_date_snow.csv", index=False)

print("Files written to:", out_dir)



## 10. Wrap‑Up

You’ve seen how to:
1. Define data modeling and its layers.  
2. Build a **Star Schema** from transactional data.  
3. Convert the star into a **Snowflake Schema**.  
4. Write DDL and queries to operate on the model.  

You can now adapt this template to domains like **O2C (Order‑to‑Cash)**, **P2P (Procure‑to‑Pay)**, marketing, finance, healthcare, etc.
